# Cell-type composition analysis

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)
library(Matrix)
library(pheatmap)


<b><font size=5 color=pink >Step 1: compare haematopoietic cell composition across references</font></b>


In [ ]:
Haematopoietic_merged <- readRDS(file.path(project_root, "data", "processed", "BMOs_Haematopoietic_Final_Integrated_CCA.rds"))


In [ ]:
DimPlot(Haematopoietic_merged, reduction = "umap", group.by = "orig.ident")


In [ ]:
table(Haematopoietic_merged$celltype, useNA = "ifany")


In [ ]:
DimPlot(Haematopoietic_merged, reduction = "umap", group.by = "celltype", label = T)


In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

# ============================================================
# ============================================================

Haematopoietic_merged$celltype <- as.character(Haematopoietic_merged$celltype)
Haematopoietic_merged$dataset <- as.character(Haematopoietic_merged$dataset)

Haematopoietic_merged$group_plot <- as.character(Haematopoietic_merged$group)

idx_na <- is.na(Haematopoietic_merged$group_plot) |
  Haematopoietic_merged$group_plot == ""

Haematopoietic_merged$group_plot[idx_na] <-
  Haematopoietic_merged$dataset[idx_na]

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Adult_BM", "AdultBM", "Adult BM")
] <- "ABM"

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("FBM", "Fetal_BM", "Fetal BM")
] <- "FBM"

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Organoids23", "BMO2023", "BMO_2023")
] <- "BMO-2023"

table(Haematopoietic_merged$group_plot, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

Haematopoietic_merged$celltype_broad <- dplyr::case_when(

  # HSPC / progenitor
  Haematopoietic_merged$celltype %in% c(
    "HSC",
    "MPP",
    "HSPC",
    "Cycling HSPC",
    "HSC/MPP and pro",
    "CLP",
    "MEP",
    "GMP",
    "Early Myeloid Progenitor",
    "Myeloid Progenitor"
  ) ~ "HSPC/Progenitor",

  # Erythroid
  Haematopoietic_merged$celltype %in% c(
    "Erythroid",
    "erythroid",
    "Erythroblast",
    "Late Erythroid",
    "RBC"
  ) ~ "Erythroid",

  # Megakaryocyte
  Haematopoietic_merged$celltype %in% c(
    "Megakaryocyte",
    "MK"
  ) ~ "Megakaryocyte",

  # Monocyte / macrophage
  Haematopoietic_merged$celltype %in% c(
    "Monocyte",
    "monocyte",
    "Macrophage",
    "Macrophages"
  ) ~ "Monocyte/Macrophage",

  # Granulocyte / mast / basophil / eosinophil
  Haematopoietic_merged$celltype %in% c(
    "Neutrophil",
    "neutrophil",
    "Basophil",
    "Eosinophil",
    "Mast",
    "Ba/Eo/Ma",
    "eo/baso/mast",
    "Late Myeloid"
  ) ~ "Granulocyte/Mast",

  # DC
  Haematopoietic_merged$celltype %in% c(
    "DC",
    "pDC",
    "Cycling DCs"
  ) ~ "DC",

  # B lineage
  Haematopoietic_merged$celltype %in% c(
    "B_lineage",
    "Pre-Pro B",
    "Pro-B",
    "Pre-B",
    "Mature B"
  ) ~ "B lineage",

  # Plasma cell
  Haematopoietic_merged$celltype %in% c(
    "Plasma Cell"
  ) ~ "Plasma cell",

  # T / NK
  Haematopoietic_merged$celltype %in% c(
    "CD4+ T-Cell",
    "CD8+ T-Cell",
    "T_NK"
  ) ~ "T/NK",

  TRUE ~ "Other"
)

table(Haematopoietic_merged$celltype, Haematopoietic_merged$celltype_broad, useNA = "ifany")
table(Haematopoietic_merged$celltype_broad, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

group_levels <- c(
  "BMO-2023",
  "Static.25d",
  "Dynamic.25d",
  "Dynamic.31d",
  "ABM",
  "FBM"
)

broad_levels <- c(
  "HSPC/Progenitor",
  "Erythroid",
  "Megakaryocyte",
  "Monocyte/Macrophage",
  "Granulocyte/Mast",
  "DC",
  "B lineage",
  "Plasma cell",
  "T/NK",
  "Other"
)

prop_df <- Haematopoietic_merged@meta.data %>%
  dplyr::filter(
    group_plot %in% group_levels,
    !is.na(celltype_broad),
    celltype_broad != "Other"
  ) %>%
  dplyr::mutate(
    group_plot = factor(group_plot, levels = group_levels),
    celltype_broad = factor(celltype_broad, levels = broad_levels)
  )

group_n <- prop_df %>%
  dplyr::count(group_plot, name = "n_group")

stack_df <- prop_df %>%
  dplyr::count(group_plot, celltype_broad, name = "n") %>%
  dplyr::left_join(group_n, by = "group_plot") %>%
  dplyr::mutate(
    freq = n / n_group,
    pct_label = paste0(round(freq * 100, 1), "%")
  )

stack_df


In [ ]:
# ============================================================
# ============================================================

outdir <- file.path(project_root, "results", "figures", "composition")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

saveRDS(
  stack_df,
  file = file.path(outdir, "Haematopoietic_broad_celltype_stack_df.rds")
)

write.csv(
  stack_df,
  file = file.path(outdir, "Haematopoietic_broad_celltype_stack_df.csv"),
  row.names = FALSE
)

saveRDS(
  group_n,
  file = file.path(outdir, "Haematopoietic_broad_celltype_group_n.rds")
)

write.csv(
  group_n,
  file = file.path(outdir, "Haematopoietic_broad_celltype_group_n.csv"),
  row.names = FALSE
)

cat("stack_df was saved to:\n\n")
cat(file.path(outdir, "Haematopoietic_broad_celltype_stack_df.rds"), "\n")
cat(file.path(outdir, "Haematopoietic_broad_celltype_stack_df.csv"), "\n")


In [ ]:
stack_df <- readRDS(
  file.path(project_root, "results", "figures", "Haematopoietic_broad_celltype_stack_df.rds")
)

group_n <- readRDS(
  file.path(project_root, "results", "figures", "Haematopoietic_broad_celltype_group_n.rds")
)

head(stack_df)
group_n


In [ ]:
# ============================================================
# ============================================================

broad_cols <- c(
  "HSPC/Progenitor"    = "#D62828",
  "Erythroid"          = "#E76F51",
  "Megakaryocyte"      = "#8C510A",
  "Monocyte/Macrophage"= "#2B6CB0",
  "Granulocyte/Mast"   = "#4DBBD5",
  "DC"                 = "#577590",
  "B lineage"          = "#1B9E77",
  "Plasma cell"        = "#006D77",
  "T/NK"               = "#A6761D",
  "Other"              = "#BDBDBD"
)

# ============================================================
# ============================================================

p_haem_prop <- ggplot(
  stack_df,
  aes(x = group_plot, y = freq, fill = celltype_broad)
) +
  geom_col(
    width = 0.68,
    color = "white",
    linewidth = 0.45
  ) +
  geom_text(
    aes(label = ifelse(freq >= 0.05, pct_label, "")),
    position = position_stack(vjust = 0.5),
    color = "white",
    size = 3.1,
    fontface = "bold"
  ) +
  geom_text(
    data = group_n,
    aes(
      x = group_plot,
      y = 1.04,
      label = paste0("n = ", n_group)
    ),
    inherit.aes = FALSE,
    size = 3.4,
    fontface = "italic",
    color = "black"
  ) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0, 0.08)),
    limits = c(0, 1.12)
  ) +
  scale_fill_manual(values = broad_cols) +
  theme_classic(base_size = 13) +
  labs(
    title = "Haematopoietic cell composition",
    subtitle = "BMO-2023 vs DBMOs vs adult and fetal bone marrow",
    x = NULL,
    y = "Proportion of cells",
    fill = "Cell type"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    plot.subtitle = element_text(
      hjust = 0.5,
      size = 11,
      color = "grey30"
    ),
    axis.text.x = element_text(
      face = "bold",
      size = 10.5,
      color = "black",
      angle = 25,
      hjust = 1
    ),
    axis.text.y = element_text(
      size = 10,
      color = "black"
    ),
    axis.title.y = element_text(
      size = 12,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.45
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "right",
    legend.title = element_text(
      face = "bold",
      size = 10,
      color = "black"
    ),
    legend.text = element_text(
      size = 9,
      color = "black"
    ),
    legend.key.size = unit(0.45, "cm")
  )

p_haem_prop


In [ ]:
broad_cols <- c(
  "HSPC/Progenitor"     = "#C75D5D",
  "Erythroid"           = "#E89A8B",
  "Megakaryocyte"       = "#B98E5A",
  "Monocyte/Macrophage" = "#5B8DB8",
  "Granulocyte/Mast"    = "#8EC6D6",
  "DC"                  = "#7C8FA8",
  "B lineage"           = "#7AAE8B",
  "Plasma cell"         = "#5A9EA0",
  "T/NK"                = "#B86B8B",
  "Other"               = "#CFCFCF"
)

p_haem_prop <- ggplot(
  stack_df,
  aes(x = group_plot, y = freq, fill = celltype_broad)
) +
  geom_col(
    width = 0.82
  ) +
  geom_text(
    aes(label = ifelse(freq >= 0.05, pct_label, "")),
    position = position_stack(vjust = 0.5),
    color = "white",
    size = 3.1
  ) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0, 0.03)),
    limits = c(0, 1)
  ) +
  scale_fill_manual(values = broad_cols) +
  theme_classic(base_size = 13) +
  labs(
    title = "Haematopoietic cell composition",
    x = NULL,
    y = "Proportion of cells",
    fill = "Cell type"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    axis.text.x = element_text(
      size = 10.5,
      color = "black",
      angle = 25,
      hjust = 1
    ),
    axis.text.y = element_text(
      size = 10,
      color = "black"
    ),
    axis.title.y = element_text(
      size = 12,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.45
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "right",
    legend.title = element_text(
      face = "bold",
      size = 10,
      color = "black"
    ),
    legend.text = element_text(
      size = 9,
      color = "black"
    ),
    legend.key.size = unit(0.45, "cm")
  )

p_haem_prop


In [ ]:
outdir <- file.path(project_root, "results", "figures", "proposion")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

ggsave(
  filename = file.path(outdir, "Haematopoietic_broad_celltype_composition.pdf"),
  plot = p_haem_prop,
  width = 6,
  height = 4.9,
  units = "in",
  device = "pdf"
)


In [ ]:
library(Seurat)
library(ggplot2)

Haematopoietic_merged$celltype_broad <- factor(
  Haematopoietic_merged$celltype_broad,
  levels = names(broad_cols)
)

p_umap_broad <- DimPlot(Haematopoietic_merged, reduction = "umap", group.by = "celltype_broad",  #label = TRUE,
         cols = broad_cols, pt.size = 1) +
  theme(aspect.ratio = 1.2)+
  labs(title = "Haematopoietic cell types")

p_umap_broad


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "Haematopoietic_celltype_broad_UMAP.pdf"),
  plot = p_umap_broad,
  width = 6.2,
  height = 5.5,
  units = "in",
  device = cairo_pdf
)


<b><font size=5 color=pink >Step 2: compare stromal cell composition across references</font></b>


In [ ]:
rm(list = ls())
gc()


In [ ]:
Stromal_merged <- readRDS(file.path(project_root, "data", "processed", "BMOs_Stromal_Final_Integrated_CCA.rds"))


In [ ]:
DimPlot(Stromal_merged, reduction = "umap", group.by = "orig.ident")


In [ ]:
DimPlot(Stromal_merged, reduction = "umap", group.by = "celltype")


In [ ]:
table(Stromal_merged$celltype, useNA = "ifany")


In [ ]:
a <- data.frame(table(Stromal_merged$celltype, useNA = "ifany"))
a


In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)

# ============================================================
# ============================================================

Stromal_merged$celltype <- as.character(Stromal_merged$celltype)
Stromal_merged$dataset <- as.character(Stromal_merged$dataset)

Stromal_merged$group_plot <- as.character(Stromal_merged$group)

idx_na <- is.na(Stromal_merged$group_plot) |
  Stromal_merged$group_plot == ""

Stromal_merged$group_plot[idx_na] <- Stromal_merged$dataset[idx_na]

Stromal_merged$group_plot[
  Stromal_merged$group_plot %in% c("Adult_BM", "AdultBM", "Adult BM")
] <- "ABM"

Stromal_merged$group_plot[
  Stromal_merged$group_plot %in% c("FBM", "Fetal_BM", "Fetal BM")
] <- "FBM"

Stromal_merged$group_plot[
  Stromal_merged$group_plot %in% c("Organoids23", "BMO2023", "BMO-2023")
] <- "mBMO"

table(Stromal_merged$group_plot, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

Stromal_merged$stromal_broad <- dplyr::case_when(

  # Adipogenic MSC / CAR-like
  Stromal_merged$celltype %in% c(
    "adipo-CAR",
    "Adipo-MSC",
    "Adipocytes pre."
  ) ~ "Adipogenic MSC",

  # MSC / CAR / mesenchymal stromal cells
  Stromal_merged$celltype %in% c(
    "MSC",
    "MSCs",
    "APOD+ MSC",
    "Fibro-MSC",
    "RNAlo MSC",
    "THY1+ MSC",
    "Proliferating MSCs"
  ) ~ "MSC",

  # Fibroblast-like stromal cells
  Stromal_merged$celltype %in% c(
    "Fibroblast",
    "arteriolar fibroblast",
    "endosteal fibroblast",
    "myofibroblast"
  ) ~ "Fibroblast",

  # Osteogenic / chondrogenic lineage
  Stromal_merged$celltype %in% c(
    "chondrocyte",
    "Chondrocytes",
    "early osteoblast",
    "Osteo-MSC",
    "osteoblast",
    "Osteoblast",
    "Osteoblasts",
    "osteoblast precursor",
    "Osteoblasts pre.",
    "Cycling Osteoblasts pre.",
    "osteochondral precursor",
    "Osteochondral pre."
  ) ~ "Osteogenic/Chondrogenic",

  # Endothelial cells
  Stromal_merged$celltype %in% c(
    "Endothelium",
    "Endothelial Cells",
    "AEC",
    "SEC",
    "immature EC",
    "proliferating EC",
    "sinusoidal EC",
    "tip EC"
  ) ~ "Endothelial",

  # Mural cells: pericytes / vascular smooth muscle cells
  Stromal_merged$celltype %in% c(
    "Pericytes",
    "VSMC"
  ) ~ "Mural cells",

  # Macrophage / osteoclast-related stromal-associated myeloid cells
  Stromal_merged$celltype %in% c(
    "erythroid macrophage",
    "monocytoid macrophage",
    "stromal macrophage",
    "osteoclast"
  ) ~ "Macrophage/Osteoclast",

  # Neural / muscle-associated cells
  Stromal_merged$celltype %in% c(
    "schwann cells",
    "muscle",
    "muscle stem cell"
  ) ~ "Neural/Muscle",

  TRUE ~ "Other"
)

table(Stromal_merged$celltype, Stromal_merged$stromal_broad, useNA = "ifany")
table(Stromal_merged$stromal_broad, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

group_levels <- c(
  "mBMO",
  "Static.25d",
  "Dynamic.25d",
  "Dynamic.31d",
  "ABM",
  "FBM"
)

stromal_broad_levels <- c(
  "Adipogenic MSC",
  "MSC",
  "Fibroblast",
  "Osteogenic/Chondrogenic",
  "Endothelial",
  "Mural cells",
  "Macrophage/Osteoclast",
  "Neural/Muscle",
  "Other"
)

prop_df <- Stromal_merged@meta.data %>%
  dplyr::filter(
    group_plot %in% group_levels,
    !is.na(stromal_broad),
    stromal_broad != "Other"
  ) %>%
  dplyr::mutate(
    group_plot = factor(group_plot, levels = group_levels),
    stromal_broad = factor(stromal_broad, levels = stromal_broad_levels)
  )

group_n <- prop_df %>%
  dplyr::count(group_plot, name = "n_group")

stack_df_stromal <- prop_df %>%
  dplyr::count(group_plot, stromal_broad, name = "n") %>%
  dplyr::left_join(group_n, by = "group_plot") %>%
  dplyr::mutate(
    freq = n / n_group,
    pct_label = paste0(round(freq * 100, 1), "%")
  )

stack_df_stromal


In [ ]:
# ============================================================
# ============================================================

stromal_broad_cols <- c(
  "Adipogenic MSC"           = "#E6C65A",
  "MSC"                      = "#8FB56A",
  "Fibroblast"               = "#D8895A",
  "Osteogenic/Chondrogenic"  = "#B9935A",
  "Endothelial"              = "#6FA4C8",
  "Mural cells"              = "#C77DAA",
  "Macrophage/Osteoclast"    = "#8D88C0",
  "Neural/Muscle"            = "#9A7B6F",
  "Other"                    = "#CFCFCF"
)

# ============================================================
# ============================================================

p_stromal_prop <- ggplot(
  stack_df_stromal,
  aes(x = group_plot, y = freq, fill = stromal_broad)
) +
  geom_col(
    width = 0.68,
    color = "white",
    linewidth = 0.45
  ) +
  geom_text(
    aes(label = ifelse(freq >= 0.05, pct_label, "")),
    position = position_stack(vjust = 0.5),
    color = "white",
    size = 3.1,
    fontface = "bold"
  ) +
  geom_text(
    data = group_n,
    aes(
      x = group_plot,
      y = 1.04,
      label = paste0("n = ", n_group)
    ),
    inherit.aes = FALSE,
    size = 3.4,
    fontface = "italic",
    color = "black"
  ) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0, 0.08)),
    limits = c(0, 1.12)
  ) +
  scale_fill_manual(values = stromal_broad_cols) +
  theme_classic(base_size = 13) +
  labs(
    title = "Stromal cell composition",
    subtitle = "BMO-2023 vs DBMOs vs adult and fetal bone marrow",
    x = NULL,
    y = "Proportion of cells",
    fill = "Stromal type"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    plot.subtitle = element_text(
      hjust = 0.5,
      size = 11,
      color = "grey30"
    ),
    axis.text.x = element_text(
      face = "bold",
      size = 10.5,
      color = "black",
      angle = 25,
      hjust = 1
    ),
    axis.text.y = element_text(
      size = 10,
      color = "black"
    ),
    axis.title.y = element_text(
      size = 12,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.45
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "right",
    legend.title = element_text(
      face = "bold",
      size = 10,
      color = "black"
    ),
    legend.text = element_text(
      size = 9,
      color = "black"
    ),
    legend.key.size = unit(0.45, "cm")
  )

p_stromal_prop


In [ ]:
# ============================================================
# ============================================================
stromal_broad_cols <- c(
  "Adipogenic MSC"           = "#E6C65A",
  "MSC"                      = "#8FB56A",
  "Fibroblast"               = "#D8895A",
  "Osteogenic/Chondrogenic"  = "#B9935A",
  "Endothelial"              = "#6FA4C8",
  "Mural cells"              = "#C77DAA",
  "Macrophage/Osteoclast"    = "#8D88C0",
  "Neural/Muscle"            = "#9A7B6F",
  "Other"                    = "#CFCFCF"
)

p_stromal_prop <- ggplot(
  stack_df_stromal,
  aes(x = group_plot, y = freq, fill = stromal_broad)
) +
  geom_col(
    width = 0.82
  ) +
  geom_text(
    aes(label = ifelse(freq >= 0.05, pct_label, "")),
    position = position_stack(vjust = 0.5),
    color = "white",
    size = 3.1
  ) +
  scale_y_continuous(
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0, 0.03)),
    limits = c(0, 1)
  ) +
  scale_fill_manual(values = stromal_broad_cols) +
  theme_classic(base_size = 13) +
  labs(
    title = "Stromal cell composition",
    x = NULL,
    y = "Proportion of cells",
    fill = "Stromal type"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    axis.text.x = element_text(
      size = 10.5,
      color = "black",
      angle = 25,
      hjust = 1
    ),
    axis.text.y = element_text(
      size = 10,
      color = "black"
    ),
    axis.title.y = element_text(
      size = 12,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.45
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "right",
    legend.title = element_text(
      face = "bold",
      size = 10,
      color = "black"
    ),
    legend.text = element_text(
      size = 9,
      color = "black"
    ),
    legend.key.size = unit(0.45, "cm")
  )

p_stromal_prop


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "Stromal_broad_celltype_composition.pdf"),
  plot = p_stromal_prop,
  width = 6.2,
  height = 5,
  units = "in",
  device = cairo_pdf
)


In [ ]:
table(Stromal_merged@meta.data$stromal_broad)


In [ ]:
library(Seurat)
library(ggplot2)

stromal_broad_cols <- c(
  "Adipogenic MSC"           = "#E6C65A",
  "MSC"                      = "#8FB56A",
  "Fibroblast"               = "#D8895A",
  "Osteogenic/Chondrogenic"  = "#B9935A",
  "Endothelial"              = "#6FA4C8",
  "Mural cells"              = "#C77DAA",
  "Macrophage/Osteoclast"    = "#8D88C0",
  "Neural/Muscle"            = "#9A7B6F",
  "Other"                    = "#CFCFCF"
)


Stromal_merged$stromal_broad <- factor(
  Stromal_merged$stromal_broad,
  levels = names(stromal_broad_cols)
)

p_umap_broad <- DimPlot(Stromal_merged, reduction = "umap", group.by = "stromal_broad", #label = TRUE,
         cols = stromal_broad_cols, pt.size = 0.5) +
  theme(aspect.ratio = 1.2)+
  labs(title = "Haematopoietic cell types")

p_umap_broad


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "Stromal_celltype_broad_UMAP.pdf"),
  plot = p_umap_broad,
  width = 6.2,
  height = 5.5,
  units = "in",
  device = cairo_pdf
)


In [ ]:
# ============================================================
# ============================================================

outdir <- file.path(project_root, "results", "figures", "proposion")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

ggsave(
  filename = file.path(outdir, "Stromal_broad_celltype_composition.pdf"),
  plot = p_stromal_prop,
  width = 9.5,
  height = 5.8,
  units = "in",
  device = "pdf"
)

write.csv(
  stack_df_stromal,
  file.path(outdir, "Stromal_broad_celltype_composition_long.csv"),
  row.names = FALSE
)

prop_wide_stromal <- stack_df_stromal %>%
  dplyr::select(group_plot, stromal_broad, freq) %>%
  tidyr::pivot_wider(
    names_from = stromal_broad,
    values_from = freq,
    values_fill = 0
  )

write.csv(
  prop_wide_stromal,
  file.path(outdir, "Stromal_broad_celltype_composition_wide.csv"),
  row.names = FALSE
)

saveRDS(
  stack_df_stromal,
  file.path(outdir, "Stromal_broad_celltype_stack_df.rds")
)

saveRDS(
  group_n,
  file.path(outdir, "Stromal_broad_celltype_group_n.rds")
)
